# RNA target swapping test

This notebook implements two standard swapping test for SMARTBind on the HARIBOSS dataset:

**1: Target-level (Local) AuROC Swap**
- Group test set by RNA target
- Compute per-target AuROC, then average across targets
- Swap = global permutation of all RNA targets per fold, K repetitions

**2: Global AuROC Swap**
- Compute a single AuROC over ALL test samples (no grouping)
- Swap = same global permutation, K repetitions

Both protocols are applied to PDB decoys.

The script can be reproduced with the pretrained weight downloaded from [here](https://drive.google.com/drive/folders/1LpImTv3_dFX1zqbP_MS0xWP8iaeQk4ZR?usp=sharing), and HARIBOSS 5-fold sequence-based dataset downloaded from [here](https://drive.google.com/file/d/1xrLZQwTwVPCxxjUTEG9HDaZtu7nhwtiR/view?usp=sharing), although the swapping results are already provided below.

In [ ]:
import sys
sys.path.append('../../../..')
from smartbind.preprocess import build_val_test_set
from smartbind.dataloader import RLDataLoader, RLDataset
from smartbind import load_smartbind_models

import tqdm
from smartbind.model.margin import sigmoid_cosine_distance_test
import torch
import numpy as np
from sklearn.metrics import roc_auc_score
from collections import defaultdict

In [ ]:
current_path = '../../../../training_data/update/hariboss_5fd_cov08_clustered_seqid_0.3.pkl'
topk_decoy = 100
batch_size = 24
num_workers = 2
device = 'cpu'
K_SWAP_REPETITIONS = 5  # Number of swap repetitions for statistical robustness
RANDOM_SEED = 42

## Helper functions

In [ ]:
def normalize_scores(scores):
    """Min-max normalize scores to [0, 1]."""
    scores = np.array(scores)
    s_min, s_max = scores.min(), scores.max()
    if s_max > s_min:
        return (scores - s_min) / (s_max - s_min)
    return scores


def compute_scores_for_pair(smartbind_model, rna_emb, match_smol_fp, decoy_fps):
    """
    Given a pre-computed RNA embedding, compute similarity scores for
    the true ligand and all decoy ligands.
    
    Returns:
        match_score: float, similarity score for the true binder
        decoy_scores: list of float, similarity scores for decoys
    """
    anchor = rna_emb.squeeze(1)
    
    # True ligand score
    match_tok = smartbind_model.inference_list_smols([match_smol_fp])[0].unsqueeze(0)
    match_score = sigmoid_cosine_distance_test(anchor, match_tok).item()
    
    # Decoy scores
    decoy_scores = []
    decoy_tok_list = smartbind_model.inference_list_smols(decoy_fps)
    for dtok in decoy_tok_list:
        dtok = dtok.unsqueeze(0)
        ds = sigmoid_cosine_distance_test(anchor, dtok).item()
        decoy_scores.append(ds)
    
    return match_score, decoy_scores


def compute_rank_percentile(match_score, decoy_scores):
    """Compute rank percentile of the true binder among decoys."""
    rank = sum(1 for ds in decoy_scores if match_score <= ds)
    return (len(decoy_scores) + 1 - rank) / (len(decoy_scores) + 1)


def compute_pair_auroc(match_score, decoy_scores):
    """Compute AUROC for a single pair (1 positive + N decoys), with normalization."""
    scores = [match_score] + decoy_scores
    labels = [1] + [0] * len(decoy_scores)
    scores_norm = normalize_scores(scores)
    return roc_auc_score(labels, scores_norm)

## Collect all RNA embeddings and scores per fold with no swap

For each fold, we:
1. Load the model
2. Iterate through all validation pairs
3. Pre-compute and cache all RNA embeddings
4. Compute original scores for all (RNA, ligand+decoy) pairs
5. Store everything indexed by RNA name for global swapping later

### PDB Decoys

In [ ]:
all_fold_results_pdb = {}  # fold_num -> dict of results

for current_fold_num in range(1, 6):
    print(f'\n{"="*60}')
    print(f'Processing fold {current_fold_num} (PDB Decoys)')
    print(f'{"="*60}')
    
    val_rna_seq_list, val_match_smol_list, val_rna_name_list, val_match_smol_name_list, \
        val_binding_index_list, val_match_smol_fp_list, train_rna_seq_list, train_match_smol_list, \
        train_rna_name_list, train_match_smol_name_list, train_binding_index_list, \
        train_match_smol_fp_list = build_val_test_set(
            fold_num=current_fold_num - 1,
            dict_path=current_path,
            topk_decoy=topk_decoy)
    
    val_dataset = RLDataset(
        rna_sequences=val_rna_seq_list,
        rna_sequences_names=val_rna_name_list,
        match_smols=val_match_smol_fp_list,
        match_smols_names=val_match_smol_name_list,
        non_binding_index_list=val_binding_index_list,
        is_val=True)
    val_dataloader = RLDataLoader(
        dataset=val_dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        if_shuffle=False)
    
    weight_path = f'../../../../revision/reproduce/fig2/splitting_performance/weights/sequenceSplit0803/fold{current_fold_num}.pth'
    smartbind_model = load_smartbind_models(
        model_path=weight_path, device=device, vs_mode=True)
    smartbind_model.eval()
    
    # Collect all data for this fold:
    # Each entry: (rna_name, rna_seq, rna_emb, match_smol_name, match_smol_fp, decoy_fps)
    fold_entries = []
    # Cache RNA embeddings by name to avoid recomputation
    rna_emb_cache = {}
    
    with torch.no_grad():
        for batch in tqdm.tqdm(val_dataloader.dataloader, desc=f'Fold {current_fold_num}'):
            rna_sequences, match_smols, decoy_smols_list, _, _, _ = batch
            
            for rna_seq, match_smol, decoy_smols in zip(
                rna_sequences, match_smols, decoy_smols_list
            ):
                if len(decoy_smols) == 0:
                    continue
                
                rna_name = rna_seq[0]
                rna_seq_str = rna_seq[1]
                match_smol_name = match_smol[0]
                match_smol_fp = match_smol[1]
                
                # Cache RNA embedding
                if rna_name not in rna_emb_cache:
                    rna_emb = smartbind_model.inference_single_rna(rna_seq_str)
                    rna_emb_cache[rna_name] = rna_emb
                
                fold_entries.append({
                    'rna_name': rna_name,
                    'rna_seq': rna_seq_str,
                    'match_smol_name': match_smol_name,
                    'match_smol_fp': match_smol_fp,
                    'decoy_fps': list(decoy_smols),
                })
    
    # ===== Original (non-swapped) evaluation =====
    rna_names = []
    original_scores = []   # list of (match_score, decoy_scores) per entry
    original_rank_percentiles = []
    original_pair_aurocs = []
    
    with torch.no_grad():
        for entry in tqdm.tqdm(fold_entries, desc='Original eval'):
            rna_emb = rna_emb_cache[entry['rna_name']]
            match_score, decoy_scores = compute_scores_for_pair(
                smartbind_model, rna_emb, entry['match_smol_fp'], entry['decoy_fps'])
            
            rna_names.append(entry['rna_name'])
            original_scores.append((match_score, decoy_scores))
            original_rank_percentiles.append(compute_rank_percentile(match_score, decoy_scores))
            original_pair_aurocs.append(compute_pair_auroc(match_score, decoy_scores))
    print(f'Original evaluation done, mean rank percentile: {np.mean(original_rank_percentiles):.4f}, mean pair AUROC: {np.mean(original_pair_aurocs):.4f}')

    # ===== Swapped evaluation (K repetitions, global permutation) =====
    unique_rna_names = list(set(rna_names))
    n_rna = len(unique_rna_names)
    print(f'Unique RNA targets in fold {current_fold_num}: {n_rna}')
    
    swap_rep_results = []  # list of K dicts, each with scores for all entries
    
    rng = np.random.RandomState(RANDOM_SEED + current_fold_num)
    
    for k in range(K_SWAP_REPETITIONS):
        # Build global permutation sigma: original_rna -> permuted_rna
        permuted_rna_names = unique_rna_names.copy()
        rng.shuffle(permuted_rna_names)
        # Ensure derangement (no fixed points)
        if n_rna > 1:
            while any(permuted_rna_names[i] == unique_rna_names[i] for i in range(n_rna)):
                rng.shuffle(permuted_rna_names)
        sigma = dict(zip(unique_rna_names, permuted_rna_names))
        
        # For each entry, use the swapped RNA embedding
        swapped_scores_k = []
        swapped_rank_percentiles_k = []
        swapped_pair_aurocs_k = []
        
        with torch.no_grad():
            for entry in fold_entries:
                swapped_rna_name = sigma[entry['rna_name']]
                swapped_rna_emb = rna_emb_cache[swapped_rna_name]
                
                match_score, decoy_scores = compute_scores_for_pair(
                    smartbind_model, swapped_rna_emb,
                    entry['match_smol_fp'], entry['decoy_fps'])
                
                swapped_scores_k.append((match_score, decoy_scores))
                swapped_rank_percentiles_k.append(compute_rank_percentile(match_score, decoy_scores))
                swapped_pair_aurocs_k.append(compute_pair_auroc(match_score, decoy_scores))
        
        swap_rep_results.append({
            'scores': swapped_scores_k,
            'rank_percentiles': swapped_rank_percentiles_k,
            'pair_aurocs': swapped_pair_aurocs_k,
        })
        print(f'  Swap repetition {k+1}/{K_SWAP_REPETITIONS} done.')
    
    all_fold_results_pdb[current_fold_num] = {
        'rna_names': rna_names,
        'original_scores': original_scores,
        'original_rank_percentiles': original_rank_percentiles,
        'original_pair_aurocs': original_pair_aurocs,
        'swap_rep_results': swap_rep_results,
        'unique_rna_names': unique_rna_names,
        'n_entries': len(fold_entries),
    }
    
    print(f'Fold {current_fold_num} done. Entries: {len(fold_entries)}')

### PDB Decoys — Target-level (Local) AuROC

In [ ]:
# For each target (rna_name), we have 1 positive + N decoys.
# Per-target AUROC = roc_auc_score on that group.
# local_auc = mean of per-target AUROCs.

print('\n' + '='*70)
print('TARGET-LEVEL (LOCAL) AuROC SWAP — PDB Decoys')
print('='*70)

all_local_auc_original = []
all_local_auc_swapped_per_k = [[] for _ in range(K_SWAP_REPETITIONS)]
all_local_rp_original = []
all_local_rp_swapped_per_k = [[] for _ in range(K_SWAP_REPETITIONS)]

for fold_num in range(1, 6):
    res = all_fold_results_pdb[fold_num]
    rna_names = res['rna_names']
    
    # --- Original: group by rna_name ---
    target_aurocs_orig = {}
    target_rps_orig = defaultdict(list)
    for i, rn in enumerate(rna_names):
        target_rps_orig[rn].append(res['original_rank_percentiles'][i])
        if rn not in target_aurocs_orig:
            target_aurocs_orig[rn] = []
        target_aurocs_orig[rn].append(res['original_pair_aurocs'][i])
    
    # Per-target mean AUROC (each target may have multiple pairs if same RNA appears multiple times)
    valid_target_aurocs_orig = []
    for rn, aurocs in target_aurocs_orig.items():
        valid_target_aurocs_orig.append(np.mean(aurocs))
    
    fold_local_auc_orig = np.mean(valid_target_aurocs_orig)
    fold_local_rp_orig = np.mean([np.mean(rps) for rps in target_rps_orig.values()])
    all_local_auc_original.append(fold_local_auc_orig)
    all_local_rp_original.append(fold_local_rp_orig)
    
    # --- Swapped: for each repetition k ---
    for k in range(K_SWAP_REPETITIONS):
        swap_res = res['swap_rep_results'][k]
        target_aurocs_swap = defaultdict(list)
        target_rps_swap = defaultdict(list)
        
        for i, rn in enumerate(rna_names):  # group by ORIGINAL rna_name
            target_aurocs_swap[rn].append(swap_res['pair_aurocs'][i])
            target_rps_swap[rn].append(swap_res['rank_percentiles'][i])
        
        valid_target_aurocs_swap = [np.mean(aurocs) for aurocs in target_aurocs_swap.values()]
        all_local_auc_swapped_per_k[k].append(np.mean(valid_target_aurocs_swap))
        all_local_rp_swapped_per_k[k].append(np.mean([np.mean(rps) for rps in target_rps_swap.values()]))
    
    print(f'Fold {fold_num}: Local AuROC original = {fold_local_auc_orig:.4f}')

# Aggregate across folds: mean ± std of fold-level local AUCs
local_auc_original = np.mean(all_local_auc_original)
local_auc_original_std = np.std(all_local_auc_original, ddof=1)
local_rp_original = np.mean(all_local_rp_original)
local_rp_original_std = np.std(all_local_rp_original, ddof=1)

# Also compute flat per-pair stats (for comparability with old notebook)
all_pair_aurocs_flat = []
all_pair_rps_flat = []
for fold_num in range(1, 6):
    res = all_fold_results_pdb[fold_num]
    all_pair_aurocs_flat.extend(res['original_pair_aurocs'])
    all_pair_rps_flat.extend(res['original_rank_percentiles'])
flat_auc_mean = np.mean(all_pair_aurocs_flat)
flat_auc_std = np.std(all_pair_aurocs_flat, ddof=1)
flat_rp_mean = np.mean(all_pair_rps_flat)
flat_rp_std = np.std(all_pair_rps_flat, ddof=1)

# For each k, compute mean across folds, then take mean/std across K
local_auc_swapped_per_k = [np.mean(all_local_auc_swapped_per_k[k]) for k in range(K_SWAP_REPETITIONS)]
local_rp_swapped_per_k = [np.mean(all_local_rp_swapped_per_k[k]) for k in range(K_SWAP_REPETITIONS)]

local_auc_swapped_mean = np.mean(local_auc_swapped_per_k)
local_auc_swapped_std = np.std(local_auc_swapped_per_k, ddof=1)
local_rp_swapped_mean = np.mean(local_rp_swapped_per_k)
local_rp_swapped_std = np.std(local_rp_swapped_per_k, ddof=1)

# Per-k performance gap, then mean ± std
gap_local_auc_per_k = [100 * (local_auc_swapped_per_k[k] - local_auc_original) / local_auc_original for k in range(K_SWAP_REPETITIONS)]
gap_local_rp_per_k = [100 * (local_rp_swapped_per_k[k] - local_rp_original) / local_rp_original for k in range(K_SWAP_REPETITIONS)]
gap_local_auc_mean = np.mean(gap_local_auc_per_k)
gap_local_auc_std = np.std(gap_local_auc_per_k, ddof=1)
gap_local_rp_mean = np.mean(gap_local_rp_per_k)
gap_local_rp_std = np.std(gap_local_rp_per_k, ddof=1)

print(f'\n=== Target-level (Local) Results — PDB Decoys ===')
print(f'Original  Local AuROC: {local_auc_original:.4f} ± {local_auc_original_std:.4f} (across 10 folds)')
print(f'Swapped   Local AuROC: {local_auc_swapped_mean:.4f} ± {local_auc_swapped_std:.4f} (K={K_SWAP_REPETITIONS})')
print(f'Performance Gap:       {gap_local_auc_mean:.2f}% ± {gap_local_auc_std:.2f}%')
print(f'\nOriginal  Local Rank Percentile: {local_rp_original:.4f} ± {local_rp_original_std:.4f} (across 10 folds)')
print(f'Swapped   Local Rank Percentile: {local_rp_swapped_mean:.4f} ± {local_rp_swapped_std:.4f}')
print(f'Performance Gap:                 {gap_local_rp_mean:.2f}% ± {gap_local_rp_std:.2f}%')
print(f'\n--- Flat per-pair stats ---')
print(f'Total pairs: {len(all_pair_aurocs_flat)}')
print(f'Flat Mean Rank Percentile: {flat_rp_mean:.4f} ± {flat_rp_std:.4f}')
print(f'Flat Mean Per-pair AUROC:  {flat_auc_mean:.4f} ± {flat_auc_std:.4f}')

### PDB Decoys — Global AuROC

In [ ]:
# Pool ALL (label, score) across all entries in a fold, compute one AuROC.

print('\n' + '='*70)
print('GLOBAL AuROC SWAP — PDB Decoys')
print('='*70)

all_global_auc_original = []
all_global_auc_swapped_per_k = [[] for _ in range(K_SWAP_REPETITIONS)]

for fold_num in range(1, 6):
    res = all_fold_results_pdb[fold_num]
    
    # --- Original: flatten all scores and labels ---
    all_labels = []
    all_scores = []
    for match_score, decoy_scores in res['original_scores']:
        all_scores.append(match_score)
        all_labels.append(1)
        for ds in decoy_scores:
            all_scores.append(ds)
            all_labels.append(0)
    
    all_scores_norm = normalize_scores(all_scores)
    fold_global_auc_orig = roc_auc_score(all_labels, all_scores_norm)
    all_global_auc_original.append(fold_global_auc_orig)
    
    # --- Swapped: for each repetition k ---
    for k in range(K_SWAP_REPETITIONS):
        swap_res = res['swap_rep_results'][k]
        all_labels_s = []
        all_scores_s = []
        for match_score, decoy_scores in swap_res['scores']:
            all_scores_s.append(match_score)
            all_labels_s.append(1)
            for ds in decoy_scores:
                all_scores_s.append(ds)
                all_labels_s.append(0)
        
        all_scores_s_norm = normalize_scores(all_scores_s)
        fold_global_auc_swap = roc_auc_score(all_labels_s, all_scores_s_norm)
        all_global_auc_swapped_per_k[k].append(fold_global_auc_swap)
    
    print(f'Fold {fold_num}: Global AuROC original = {fold_global_auc_orig:.4f}')

# Aggregate: mean across folds for each k, then mean/std across K
global_auc_original = np.mean(all_global_auc_original)
global_auc_swapped_per_k = [np.mean(all_global_auc_swapped_per_k[k]) for k in range(K_SWAP_REPETITIONS)]
global_auc_swapped_mean = np.mean(global_auc_swapped_per_k)
global_auc_swapped_std = np.std(global_auc_swapped_per_k, ddof=1)

# Per-k performance gap, then mean ± std
gap_global_auc_per_k = [100 * (global_auc_swapped_per_k[k] - global_auc_original) / global_auc_original for k in range(K_SWAP_REPETITIONS)]
gap_global_auc_mean = np.mean(gap_global_auc_per_k)
gap_global_auc_std = np.std(gap_global_auc_per_k, ddof=1)

print(f'\n=== Global Results — PDB Decoys ===')
print(f'Original  Global AuROC: {global_auc_original:.4f}')
print(f'Swapped   Global AuROC: {global_auc_swapped_mean:.4f} ± {global_auc_swapped_std:.4f} (K={K_SWAP_REPETITIONS})')
print(f'Performance Gap:        {gap_global_auc_mean:.2f}% ± {gap_global_auc_std:.2f}%')

In [ ]:
print('\n' + '='*90)
print('SUMMARY TABLE')
print('='*90)
print(f'{"":<35} {"Original":>12} {"Swapped":>20} {"Gap (%)":>18}')
print('-'*90)

# PDB Decoys
print(f'{"PDB — Target-level AuROC":<35} {local_auc_original:>12.4f} {local_auc_swapped_mean:>10.4f} ± {local_auc_swapped_std:.4f} {gap_local_auc_mean:>8.2f} ± {gap_local_auc_std:.2f}')
print(f'{"PDB — Target-level Rank %":<35} {local_rp_original:>12.4f} {local_rp_swapped_mean:>10.4f} ± {local_rp_swapped_std:.4f} {gap_local_rp_mean:>8.2f} ± {gap_local_rp_std:.2f}')
print(f'{"PDB — Global AuROC":<35} {global_auc_original:>12.4f} {global_auc_swapped_mean:>10.4f} ± {global_auc_swapped_std:.4f} {gap_global_auc_mean:>8.2f} ± {gap_global_auc_std:.2f}')
print()